In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

from grpo_abstr import (
    compute_advantages,
    compute_group_rewards,
    compute_ratio,
    completion_token_logprobs,
    extract_gold_answer,
    extract_prediction,
    sample_group_completions,
)

# 1. Configuration
MODEL_NAME = "Qwen/Qwen3-0.6B"
DATASET_NAME = "openai/gsm8k"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

dtype = (
    torch.bfloat16 if device.type == "cuda"
    else torch.float16 if device.type == "mps"
    else torch.float32
)


In [2]:
print("Device:", device)

# 2. Load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
).to(device).eval()


Device: mps


In [3]:
# 3. Load GSM8K
train_data = load_dataset(
    DATASET_NAME, "main", split="train[:64]"
)


In [4]:
eval_data = load_dataset(
    DATASET_NAME, "main", split="test[:32]"
)

print("Train:", len(train_data))
print("Eval:", len(eval_data))
print(train_data[0])

Train: 64
Eval: 32
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [ ]:
# 4. Prompt formatting
def make_prompt(question):
    return (
        "Solve the following math problem.\n"
        "Return only the final numerical answer.\n"
        "Use the format: FINAL_ANSWER: <number>\n\n"
        f"Question: {question}"
    )

example = train_data[0]


In [ ]:
conversations = [
    [{"role": "user", "content": make_prompt(record["question"])}]
    for record in train_data
]


In [ ]:
tokenizer.padding_side = "left"
model_inputs = tokenizer.apply_chat_template(
    conversations,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    padding=True,
    return_dict=True,
    return_tensors="pt",
)
input_ids = model_inputs["input_ids"].to(device)
attention_mask = model_inputs["attention_mask"].to(device)


In [ ]:
group_size = 4
output_ids, predictions, sequence_attention_mask = sample_group_completions(
    model=model,
    tokenizer=tokenizer,
    input_ids=input_ids,
    attention_mask=attention_mask,
    group_size=group_size,
    max_new_tokens=128,
    temperature=0.9,
)


In [ ]:
print("\nQuestion:", train_data[0]["question"])
print("\nGround truth:", train_data[0]["answer"])
print("\nPrediction:", predictions[0])


In [ ]:
print(f"Extracted Answer: {extract_gold_answer(train_data[0]['answer'])}")
print(f"Predicted Answer: {extract_prediction(predictions[0])}")


In [ ]:
gold_answers = [record["answer"] for record in train_data]
rewards = compute_group_rewards(
    predictions=predictions,
    gold_answers=gold_answers,
    group_size=group_size,
    device=device,
)


In [21]:
rewards

tensor([[0., 0., 0., 0.]], device='mps:0')

In [ ]:
advantages = compute_advantages(rewards)
advantages.shape

### Reward Advantage

In [ ]:
prompt_len = input_ids.shape[1]
with torch.no_grad():
    logits = model(
        output_ids,
        attention_mask=sequence_attention_mask,
        use_cache=False,
    ).logits

token_logps, mask = completion_token_logprobs(
    logits,
    output_ids,
    prompt_len=prompt_len,
    attention_mask=sequence_attention_mask,
)

print("Logprob shape:", token_logps.shape)
print("Mask shape:", mask.shape)
print("Completion tokens in first sequence:", mask.sum(dim=1)[0].item())
print("Sequence logprob:", token_logps[mask].sum().item())
assert torch.all(mask.sum(dim=1) > 0)


In [ ]:
# Old policy: gradients disabled
with torch.no_grad():
    old_logits = model(
        output_ids,
        attention_mask=sequence_attention_mask,
        use_cache=False,
    ).logits
    old_logps, mask = completion_token_logprobs(
        old_logits,
        output_ids,
        prompt_len,
        attention_mask=sequence_attention_mask,
    )


In [ ]:
print(predictions[0])
print("Inference tensor:", output_ids.is_inference())


In [ ]:
# generated_ids = model.generate(
#     output_ids,
#     max_new_tokens=128,
#     do_sample=False,
# )

# answer_ids = generated_ids[0, output_ids.shape[1]:]
# print(tokenizer.decode(answer_ids, skip_special_tokens=True).strip())

In [ ]:
# Current policy: gradients enabled
current_output = model(
    output_ids,
    attention_mask=sequence_attention_mask,
    use_cache=False,
)
current_logits = current_output.logits


In [ ]:
current_logps, _ = completion_token_logprobs(
    current_logits,
    output_ids,
    prompt_len,
    attention_mask=sequence_attention_mask,
)


In [ ]:
# Compute probability ratio
ratio = compute_ratio(current_logps, old_logps)

print("Old requires grad:", old_logps.requires_grad)
print("Current requires grad:", current_logps.requires_grad)
print("Ratios:", ratio[mask])
print("Mean ratio:", ratio[mask].mean().item())

In [ ]:
group_advantages = advantages.reshape(-1, 1)
unclipped = ratio * group_advantages
clipped = torch.clamp(ratio, 0.8, 1.2) * group_advantages

loss = -torch.minimum(unclipped, clipped)


In [ ]:
def grpo_policy_loss(
    current_logps,
    old_logps,
    advantages,
    mask,
    eps=0.2,
):
    """
    current_logps: (N, T)
    old_logps:     (N, T)
    advantages:    (N,)
    mask:          (N, T), boolean

    N = number of completions
    T = aligned token positions
    """

    # 1. Probability ratio
    ratio = torch.exp(
        current_logps.float() - old_logps.detach().float()
    )

    # 2. Broadcast advantages across tokens
    adv = advantages.detach().reshape(-1, 1)

    # 3. PPO clipped objective
    unclipped = ratio * adv

    clipped = torch.clamp(
        ratio, 1 - eps, 1 + eps
    ) * adv

    # 4. Negative because optimizer minimizes loss
    token_loss = -torch.minimum(unclipped, clipped)

    # 5. Ignore prompt/padding tokens
    loss = (token_loss * mask).sum() / mask.sum().clamp_min(1)

    return loss